In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from chains.extract_chain import extract_chain
from chains.match_chain import match_chain
from chains.score_chain import score_chain
from chains.explain_chain import explain_chain

In [3]:
def load_file(path):
    with open(path, "r") as f:
        return f.read()

jd = load_file("data/job_description.txt")

In [4]:
import json
import re

def run_pipeline(resume_path):
    resume = load_file(resume_path)

    # Step 1: Extract
    resume_data = extract_chain.invoke(
        {"resume": resume},
        config={"run_name": "extract"}
    )

    # Step 2: Match
    match_data = match_chain.invoke(
        {
            "resume_data": resume_data,
            "job_description": jd
        },
        config={"run_name": "match"}
    )

    # Step 3: Score
    score_raw = score_chain.invoke(
        {"match_data": match_data},
        config={"run_name": "score"}
    )

    print("RAW SCORE OUTPUT:", score_raw)

    # Clean markdown ```json ```
    cleaned = re.sub(r"```json|```", "", score_raw).strip()

    try:
        score = json.loads(cleaned)["score"]
    except:
        match = re.search(r"\d{1,3}", cleaned)
        score = int(match.group()) if match else 0

    # Step 4: Explain
    explanation = explain_chain.invoke(
        {
            "score": score,
            "match_data": match_data
        },
        config={"run_name": "explain"}
    )

    print("\n===== RESULT =====")
    print("Score:", score)
    print("Explanation:", explanation)

    return score, explanation

from langchain_core.runnables import RunnableLambda

pipeline = RunnableLambda(
    lambda resume_path: run_pipeline(resume_path)
).with_config({"run_name": "resume_pipeline"})

In [5]:
print("Strong Candidate")
pipeline.invoke("data/strong_resume.txt")

print("\nAverage Candidate")
pipeline.invoke("data/average_resume.txt")

print("\nWeak Candidate")
pipeline.invoke("data/weak_resume.txt")

Strong Candidate
RAW SCORE OUTPUT: {
  "score": 90
}

===== RESULT =====
Score: 90
Explanation: **Candidate Evaluation**

**Score:** 90

However, since the score is 90 and the match percentage is 100 according to the match data provided, there might be an issue with the scoring system. Nevertheless, we will proceed with the given data.

**Strengths:**

- The candidate has a strong match with the required skills, as all 5 skills ("Python", "Machine Learning", "SQL", "Pandas", "Scikit-learn") are present in their resume.
- The candidate's resume perfectly matches the required skills, indicating a high level of relevance.

**Missing Skills:**

- There are no missing skills mentioned in the match data, as all required skills were found in the candidate's resume.

**Final Reason:**

- Given the perfect match and the absence of any missing skills, one might expect a higher score. However, since the score is 90, it's likely that there are other factors not related to this match data that have

(0,
 "**Candidate Evaluation:**\n\n**Strengths:**\n\n* There are no strengths to highlight, as the candidate's resume does not contain any relevant skills that match the job requirements.\n\n**Missing Skills:**\n\n* The candidate is lacking in all the required skills for the job:\n  - Python\n  - Machine Learning\n  - SQL\n  - Pandas\n  - Scikit-learn\n\n**Final Reason:**\n\nThe candidate's resume does not demonstrate any relevant skills that match the job requirements, resulting in a 0% match percentage. Therefore, the candidate's application does not meet the necessary qualifications for the position.")